## Notebook 2 - Data Cleaning

## Goal:
Clean and prepare the dataset for analysis by:
- Renaming columns for better readability
- Removing unnecessary identifier columns
- Verifying data quality
- Creating a SQLite database
- Loading cleaned data into the database
- Saving the processed dataset

## 1.Import Libraries

In [5]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
import os
os.chdir('..')
print(os.getcwd())

warnings.filterwarnings('ignore')

print("Libraries Imported Successfully")

/Users/jayeshranghera/Documents/Projects/predictive-maintenance-analysis
Libraries Imported Successfully


## 2.Load Raw Data

In [7]:
df = pd.read_csv('data/raw/ai4i2020.csv')
print("Raw Data Load Successfully")
print(f'Dataset Shape: {df.shape}')

Raw Data Load Successfully
Dataset Shape: (10000, 14)


## 3.Rename Coulumns
Renaming columns to make them more readable and consistent with Python naming conventions.

In [9]:
# Define column name mapping
column_mapping = {
    'UDI': 'udi',
    'Product ID': 'product_id',
    'Type': 'machine_type',
    'Air temperature [K]': 'air_temp_k',
    'Process temperature [K]': 'process_temp_k',
    'Rotational speed [rpm]': 'rotational_speed_rpm',
    'Torque [Nm]': 'torque_nm',
    'Tool wear [min]': 'tool_wear_min',
    'Machine failure': 'machine_failure',
    'TWF': 'tool_wear_failure',
    'HDF': 'heat_dissipation_failure',
    'PWF': 'power_failure',
    'OSF': 'overstrain_failure',
    'RNF': 'random_failure'
}

#Rename Coulumns
df.rename(columns=column_mapping, inplace=True)

print(f'\n New Column Names:')
for i, col in enumerate(df.columns, 1):
    print(f'   {i}. {col}')


 New Column Names:
   1. udi
   2. product_id
   3. machine_type
   4. air_temp_k
   5. process_temp_k
   6. rotational_speed_rpm
   7. torque_nm
   8. tool_wear_min
   9. machine_failure
   10. tool_wear_failure
   11. heat_dissipation_failure
   12. power_failure
   13. overstrain_failure
   14. random_failure


## 4. Remove Unnecessary Columns

Removing identifier columns that won't be used in analysis:
- `udi` — Sequential ID (not useful for analysis)
- `product_id` — Unique product identifier (not a feature)

In [11]:
columns_to_remove = ['udi','product_id']

df_clean = df.drop(columns=columns_to_remove)

print("Columns Removed Successfully")
print(f'New Shape: {df_clean.shape}')

print("\n Remaining Columns: ")
for i, col in enumerate(df_clean.columns, 1):
    print(f' {i},{col}')

Columns Removed Successfully
New Shape: (10000, 12)

 Remaining Columns: 
 1,machine_type
 2,air_temp_k
 3,process_temp_k
 4,rotational_speed_rpm
 5,torque_nm
 6,tool_wear_min
 7,machine_failure
 8,tool_wear_failure
 9,heat_dissipation_failure
 10,power_failure
 11,overstrain_failure
 12,random_failure


## 5.Varify DataTypes

In [14]:
print("Data Types:")
print(df_clean.dtypes)

print("\n Data Type Summery")
print(f'Numerical (Int/Float): {df_clean.select_dtypes(include=['int64', 'float64']).shape[1]}')
print(f'Categorical (object): {df_clean.select_dtypes(include=['object']).shape[1]}')

Data Types:
machine_type                 object
air_temp_k                  float64
process_temp_k              float64
rotational_speed_rpm          int64
torque_nm                   float64
tool_wear_min                 int64
machine_failure               int64
tool_wear_failure             int64
heat_dissipation_failure      int64
power_failure                 int64
overstrain_failure            int64
random_failure                int64
dtype: object

 Data Type Summery
Numerical (Int/Float): 11
Categorical (object): 1


## 6.Check & Handle Duplicates

In [18]:
#check for duplicates
duplicates = df_clean.duplicated().sum()
print(f'Duplicate Rows: {duplicates}')

if duplicates == 0:
    print("No Duplicate Rows Found")
else:
    print(f"Found {duplicates} Duplicates Rows")
    print('   Action: Removing duplicates...')
    df_clean = df_clean.drop_duplicates()
    print(f' Duplicates removed. New shape: {df_clean.shape}')


Duplicate Rows: 0
No Duplicate Rows Found


## 7.Documnet Class Imbalance

**Note:** Class imbalance exists in the target variable (`machine_failure`). This is expected in real-world manufacturing data where failures are rare events. For this **data analyst** project, we will document this but not apply resampling techniques (which would be done in ML engineering).

In [19]:
# Analyse Class Distribution
failure_counts = df_clean['machine_failure'].value_counts().sort_index()
failure_pct = (failure_counts / len(df_clean) * 100).round(2)

print("\n Machine Failure Distribution: ")
print(f'No Failure(0): {failure_counts[0]:,} ({failure_pct[0]}%)')
print(f'Failure(1): {failure_counts[1]:,} ({failure_pct[1]}%)')



 Machine Failure Distribution: 
No Failure(0): 9,661 (96.61%)
Failure(1): 339 (3.39%)


Note: This imbalance reflects real-world scenarios where machine failures are rare events. This will be considered during analysis.

## 8. Create SQLite Database

Creating a SQLite database to store the cleaned data. This allows for efficient querying and demonstrates SQL skills.

In [22]:
#Databse file path
db_path = 'data/database/predictive_maintenance.db'

print(f'Database Path: {db_path}')

# Creating SQlite connection
conn = sqlite3.connect(db_path)
print("Database Connection Established")

Database Path: data/database/predictive_maintenance.db
Database Connection Established


## 9.Load Data into Database

In [ ]:
#Table Name
table_name = 'equipment_data'
print(f'Loading Data into table: {table_name}')
print(f'Taotal Records to Load: {len(df_clean)}')

#load data into SQlite
df_clean.to_sql(table_name, conn, if_exists='replace', index=False)
print("Data Loaded Successfully")

Loading Data into table: equipment_data
Taotal Records to Load: 10000
Data Loaded Successfully


## 10.Varify Database Content

In [30]:
#query 1 - check table exist
query_table = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query_table,conn)
print(f'Tables in Database:\n {tables}')

#query 2 - count total records
query_count = f"SELECT COUNT(*) AS total_records FROM {table_name};"
record_count = pd.read_sql(query_count,conn)

print(f'\n Total Records in {table_name}: {record_count["total_records"][0]:,}')

Tables in Database:
              name
0  equipment_data

 Total Records in equipment_data: 10,000


In [38]:
#query 3 - view first 10 rows
query_sample = f"SELECT * FROM {table_name} LIMIT 10;" 
sample_data = pd.read_sql(query_sample,conn)
print(f"First 10 Rows\n")
sample_data

First 10 Rows



,machine_type,air_temp_k,process_temp_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,tool_wear_failure,heat_dissipation_failure,power_failure,overstrain_failure,random_failure
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0
5,M,298.1,308.6,1425,41.9,11,0,0,0,0,0,0
6,L,298.1,308.6,1558,42.4,14,0,0,0,0,0,0
7,L,298.1,308.6,1527,40.2,16,0,0,0,0,0,0
8,M,298.3,308.7,1667,28.6,18,0,0,0,0,0,0
9,M,298.5,309.0,1741,28.0,21,0,0,0,0,0,0


In [42]:
#query 4 = check columns name and types
query_schema = f"PRAGMA table_info({table_name});"
schema = pd.read_sql(query_schema, conn)

print(f' Database Schema for {table_name}:\n')
print(schema[['name', 'type']].to_string(index=False))

 Database Schema for equipment_data:

                    name    type
            machine_type    TEXT
              air_temp_k    REAL
          process_temp_k    REAL
    rotational_speed_rpm INTEGER
               torque_nm    REAL
           tool_wear_min INTEGER
         machine_failure INTEGER
       tool_wear_failure INTEGER
heat_dissipation_failure INTEGER
           power_failure INTEGER
      overstrain_failure INTEGER
          random_failure INTEGER


In [43]:
#close database connection
conn.close()
print("Databse Connection is Closed.")

Databse Connection is Closed.


## 11.Saved Cleaned CSV File

In [45]:
# Save cleaned data to CSV
output_path = 'data/processed/cleaned_data.csv'

print(f'Saving cleaned data to: {output_path}')

df_clean.to_csv(output_path, index=False)

print('\n Cleaned data saved successfully!')
print(f' File Size: {len(df_clean):,} rows , {len(df_clean.columns)} columns')

Saving cleaned data to: data/processed/cleaned_data.csv

 Cleaned data saved successfully!
 File Size: 10,000 rows , 12 columns


# Data Cleaning Summary

---

**COMPLETED TASKS**

1. **Column Renaming**  
   - All columns renamed to **snake_case** format  
   - Units included in column names for better clarity  

2. **Data Cleaning**  
   - Removed identifier columns: `udi`, `product_id`  
   - Verified: **No missing values**  
   - Checked for duplicates

3. **Class Imbalance Documentation**  
   - Machine failure rate
   - Imbalance documented for future modeling consideration  

4. **Database Creation**  
   - SQLite database created: `predictive_maintenance.db`  
   - Table name: `equipment_data`  
   - Records loaded

5. **File Export**  
   - Cleaned dataset saved: `data/processed/cleaned_data.csv`  
   - Final shape Check
